In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_parquet('mart_occupancy_features.parquet')
print(f'Shape: {df_raw.shape}')
print(f'Rango de fechas: {df_raw["date"].min()} → {df_raw["date"].max()}')

Shape: (3327493, 34)
Rango de fechas: 2025-06-23 00:00:00 → 2026-05-15 00:00:00


In [2]:
train_end = '2026-02-28'
test_start = '2026-03-01'

df_train = df_raw[df_raw['date'] <= train_end].copy()
df_test  = df_raw[df_raw['date'] >= test_start].copy()

print(f'Train: {df_train.shape[0]:,} filas  ({df_train["date"].min().date()} → {df_train["date"].max().date()})')
print(f'Test : {df_test.shape[0]:,} filas  ({df_test["date"].min().date()} → {df_test["date"].max().date()})')
print(f'\nDistribución target train: {df_train["is_occupied"].mean():.3f}')
print(f'Distribución target test : {df_test["is_occupied"].mean():.3f}')

Train: 2,394,745 filas  (2025-06-23 → 2026-02-28)
Test : 932,748 filas  (2026-03-01 → 2026-05-15)

Distribución target train: 0.394
Distribución target test : 0.322


In [3]:
df_train['has_reviews'] = df_train['review_scores_rating'].notna().astype(int)
df_test['has_reviews']  = df_test['review_scores_rating'].notna().astype(int)

In [4]:
FEATURES = [
    'month', 'day_of_week', 'is_weekend', 'days_to_next_holiday', 'is_holiday',
    'neighbourhood_cleansed', 'room_type', 'accommodates', 'listing_price',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating', 'has_reviews',
    'instant_bookable', 'temp_mean', 'precipitation_mm',
    'num_sports', 'num_festivals', 'total_attendance',
]

TARGET = 'is_occupied'

keep = FEATURES + [TARGET]
cols_dropped = [c for c in df_train.columns if c not in keep]
df_train = df_train[keep].copy()
df_test  = df_test[keep].copy()

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')

Train: (2394745, 20)  |  Test: (932748, 20)


In [5]:
# ── Temporales: cíclicas + capping ───────────────────────────────────────────
import numpy as np

for df in [df_train, df_test]:
    # Codificación cíclica
    df['month_sin']       = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']       = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']         = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']         = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # Capping days_to_next_holiday
    df['days_to_next_holiday'] = df['days_to_next_holiday'].clip(upper=14)

In [6]:
# ── Log1p en variables con skew + interacciones de precio ────────────────────
for df in [df_train, df_test]:
    df['log_price']          = np.log1p(df['listing_price'])
    df['log_reviews']        = np.log1p(df['number_of_reviews'])
    df['log_min_nights']     = np.log1p(df['minimum_nights'])
    df['price_per_person']   = df['listing_price'] / df['accommodates'].replace(0, 1)

# price_vs_neighborhood_mean: media calculada SÓLO sobre train para evitar leakage
neigh_mean_price = df_train.groupby('neighbourhood_cleansed')['listing_price'].mean()
for df in [df_train, df_test]:
    df['price_vs_neigh_mean'] = df['listing_price'] / df['neighbourhood_cleansed'].map(neigh_mean_price).fillna(1)

In [7]:
# ── Flags de eventos + bins de asistencia y clima ────────────────────────────
for df in [df_train, df_test]:
    # Flags binarios
    df['has_sports_event'] = (df['num_sports'] > 0).astype(int)
    df['has_festival']     = (df['num_festivals'] > 0).astype(int)

    # Log de asistencia
    df['log_attendance']   = np.log1p(df['total_attendance'])

In [8]:
# ── Features finales exp_06 ───────────────────────────────────────────────────
FEATURES = [
    # Temporales (cíclicas sustituyen a month y day_of_week)
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'is_weekend', 'days_to_next_holiday', 'is_holiday',
    # Listing
    'neighbourhood_cleansed', 'room_type', 'accommodates',
    'log_price', 'price_per_person', 'price_vs_neigh_mean',
    'log_min_nights',
    # Reseñas
    'log_reviews', 'review_scores_rating', 'has_reviews',
    # Reserva
    'instant_bookable',
    # Clima
    'temp_mean', 'precipitation_mm',
    # Eventos
    'num_sports', 'num_festivals',
    'log_attendance', 'has_sports_event', 'has_festival',
]

TARGET = 'is_occupied'

num_cols = [
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'is_weekend', 'days_to_next_holiday', 'is_holiday',
    'accommodates', 'log_price', 'price_per_person', 'price_vs_neigh_mean',
    'log_min_nights',
    'log_reviews', 'review_scores_rating', 'has_reviews',
    'instant_bookable',
    'temp_mean',
    'num_sports', 'num_festivals',
    'log_attendance', 'has_sports_event', 'has_festival',
]

cat_cols = ['neighbourhood_cleansed', 'room_type']

keep = FEATURES + [TARGET]
df_train = df_train[keep].copy()
df_test  = df_test[keep].copy()

print(f'Total features: {len(FEATURES)}')
print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')

Total features: 25
Train: (2394745, 26)  |  Test: (932748, 26)


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, cat_cols),
], remainder='passthrough')

X_train = df_train[FEATURES]
y_train = df_train[TARGET].astype(int)
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET].astype(int)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'Train: {X_train_processed.shape}')
print(f'Test : {X_test_processed.shape}')

Train: (2394745, 94)
Test : (932748, 94)


In [21]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import numpy as np

optuna.logging.set_verbosity(optuna.logging.WARNING)

ratio = float((y_train == 0).sum() / (y_train == 1).sum())

def objective(trial):
    param = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight':  ratio,
        'eval_metric':       'logloss',
        'random_state':      42,
        'n_jobs':            -1,
    }

    model = XGBClassifier(**param)
    cv    = TimeSeriesSplit(n_splits=5)
    score = cross_val_score(model, X_train_processed, y_train,
                            cv=cv, scoring='f1', n_jobs=1)
    return score.mean()

study = optuna.create_study(direction='maximize')
print("Iniciando búsqueda de hiperparámetros con Optuna (100 trials, timeout 2h)...")
study.optimize(objective, n_trials=100, timeout=7200, show_progress_bar=True)

print(f"\nMejor F1-Score CV: {study.best_value:.4f}")
print("Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")


Iniciando búsqueda de hiperparámetros con Optuna (100 trials, timeout 2h)...


Best trial: 0. Best value: 0.783428:   9%|▉         | 9/100 [15:44<2:39:09, 104.94s/it, 902.69/7200 seconds]


[W 2026-05-06 17:00:59,975] Trial 9 failed with parameters: {'n_estimators': 691, 'max_depth': 8, 'learning_rate': 0.028698700330664784, 'subsample': 0.6247611383946994, 'colsample_bytree': 0.8271964010784254, 'reg_alpha': 0.0006755641121335047, 'reg_lambda': 0.07002644015199949} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/marta/Documents/Máster/IADATA-PROJECT3/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/zt/nr4slqvn18j6z9rshfmngv300000gn/T/ipykernel_63730/900350665.py", line 27, in objective
    score = cross_val_score(model, X_train_processed, y_train,
                            cv=cv, scoring='f1', n_jobs=1)
  File "/Users/marta/Documents/Máster/IADATA-PROJECT3/.venv/lib/python3.13/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "/Users/marta/Documents/Máster/IADATA

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc, RocCurveDisplay

best_params = study.best_params.copy()
best_params.update({
    'scale_pos_weight': ratio,
    'eval_metric':      'logloss',
    'random_state':     42,
    'n_jobs':           -1,
})

best_model = XGBClassifier(**best_params)
best_model.fit(X_train_processed, y_train)

y_pred  = best_model.predict(X_test_processed)
y_proba = best_model.predict_proba(X_test_processed)[:, 1]

print(classification_report(y_test, y_pred, target_names=['libre', 'ocupado'], zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=axes[0], cmap='Blues',
    colorbar=False, display_labels=['libre', 'ocupado']
)
axes[0].set_title('Matriz de confusión')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=axes[1], cmap='Blues',
    colorbar=False, normalize='true', display_labels=['libre', 'ocupado']
)
axes[1].set_title('Matriz de confusión (normalizada)')

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc).plot(ax=axes[2])
axes[2].set_title(f'Curva ROC — AUC = {roc_auc:.4f}')

plt.tight_layout()
plt.show()


In [ ]:
feature_names = preprocessor.get_feature_names_out()

fi_df = pd.DataFrame({
    'feature':    feature_names,
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=False)

fi_df.head(15).plot(
    kind='barh', x='feature', y='importance',
    figsize=(8, 6), title='Feature importances — top 15', legend=False
)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import learning_curve

cv = TimeSeriesSplit(n_splits=5)

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_processed, y_train,
    cv=cv, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1, shuffle=True, random_state=42
)

tm, ts = train_scores.mean(axis=1), train_scores.std(axis=1)
vm, vs = val_scores.mean(axis=1),   val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_sizes, tm, 'o-', color='purple',    label='Train')
ax.plot(train_sizes, vm, 'o-', color='royalblue', label='CV')
ax.fill_between(train_sizes, tm-ts, tm+ts, alpha=0.1, color='purple')
ax.fill_between(train_sizes, vm-vs, vm+vs, alpha=0.1, color='royalblue')
ax.set(xlabel='Train size', ylabel='F1', title='Learning curve')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Gap final (F1 train − CV): {tm[-1] - vm[-1]:.4f}')
